# 📊 Análisis Exploratorio — Desigualdad Urbana en Córdoba
### DiploDatos 2026 — FAMAF / Universidad Nacional de Córdoba

En este notebook vas a explorar el dataset de desigualdad urbana de los barrios de Córdoba. El objetivo es entender qué variables tenemos, cómo se distribuyen y qué patrones iniciales podemos observar.

**Dataset:** `data/processed/dataset_final_v6.csv`
- 494 barrios de la ciudad de Córdoba
- Variables censales: población, hogares, NBI
- Variables de servicios: escuelas (total/estatal/privado), centros de salud, transporte, luminarias, comisarías


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.style.use('seaborn-v0_8-whitegrid')

# Cargar dataset
df = pd.read_csv('../data/processed/dataset_final_v6.csv')
print(f'Filas: {len(df)} | Columnas: {df.shape[1]}')
df.head()

## 1. Resumen estadístico

In [ ]:
# Descripción de variables numéricas
df.describe().round(2)

In [ ]:
# Datos faltantes
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(1)
tabla_nulos = pd.DataFrame({'faltantes': nulos, 'porcentaje': pct_nulos})
tabla_nulos[tabla_nulos['faltantes'] > 0]

## 2. Distribución del NBI — ¿Cuánto varía la pobreza entre barrios?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma de pct_nbi
df['pct_nbi'].dropna().hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de % NBI por barrio', fontsize=13, fontweight='bold')
axes[0].set_xlabel('% Hogares con NBI')
axes[0].set_ylabel('Cantidad de barrios')
media = df['pct_nbi'].mean()
axes[0].axvline(media, color='red', linestyle='--', label=f'Media: {media:.1f}%')
axes[0].legend()

# Boxplot
df['pct_nbi'].dropna().plot.box(ax=axes[1], vert=True, 
    patch_artist=True, boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Boxplot % NBI', fontsize=13, fontweight='bold')
axes[1].set_ylabel('% Hogares con NBI')

plt.tight_layout()
plt.savefig('../tmp/exploracion_nbi.png', dpi=130, bbox_inches='tight')
plt.show()
print(f'Media: {df["pct_nbi"].mean():.1f}% | Mediana: {df["pct_nbi"].median():.1f}% | Máx: {df["pct_nbi"].max():.1f}%')

## 3. ¿Cuáles son los barrios con mayor y menor NBI?

In [ ]:
# Top 15 con más NBI (más vulnerables)
top_nbi = df.nlargest(15, 'pct_nbi')[['barrio', 'poblacion', 'pct_nbi']].reset_index(drop=True)
top_nbi.index = top_nbi.index + 1
print('🔴 TOP 15 barrios con mayor % NBI (más vulnerables):')
print(top_nbi.to_string())

print()

# Bottom 15 con menos NBI (menos vulnerables)
bot_nbi = df[df['pct_nbi'].notna()].nsmallest(15, 'pct_nbi')[['barrio', 'poblacion', 'pct_nbi']].reset_index(drop=True)
bot_nbi.index = bot_nbi.index + 1
print('🟢 TOP 15 barrios con menor % NBI (menos vulnerables):')
print(bot_nbi.to_string())

## 4. Cobertura de servicios — ¿Qué porcentaje de barrios tiene cada servicio?

In [ ]:
servicios = {
    'Escuelas (total)': 'escuelas_total',
    'Escuelas estatales': 'escuelas_estatales',
    'Escuelas privadas': 'escuelas_privadas',
    'Escuelas municipales': 'escuelas_municipales',
    'Centros de salud': 'centros_salud',
    'Paradas colectivo': 'paradas_colectivo',
    'Luminarias (reportes)': 'luminarias_reportes',
    'Comisarías': 'comisarias',
}

cobertura = {nombre: (df[col] > 0).mean() * 100 for nombre, col in servicios.items() if col in df.columns}
cobertura_s = pd.Series(cobertura).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(cobertura_s.index, cobertura_s.values, 
               color=['#2ecc71' if v >= 50 else '#e74c3c' if v < 20 else '#f39c12' for v in cobertura_s.values])
for bar, val in zip(bars, cobertura_s.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.0f}%', va='center', fontsize=10)
ax.set_xlabel('% de barrios con al menos 1 unidad del servicio')
ax.set_title('Cobertura de servicios urbanos\n(% de los 494 barrios)', fontsize=13, fontweight='bold')
ax.set_xlim(0, 110)
plt.tight_layout()
plt.savefig('../tmp/exploracion_cobertura.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. ¿Las escuelas privadas se concentran en barrios de bajo NBI?

In [ ]:
df_con_datos = df.dropna(subset=['pct_nbi'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Escuelas estatales vs NBI
axes[0].scatter(df_con_datos['pct_nbi'], df_con_datos['escuelas_estatales'],
                alpha=0.5, color='#3498db', s=30)
axes[0].set_xlabel('% NBI del barrio')
axes[0].set_ylabel('Cantidad de establecimientos estatales')
axes[0].set_title('Escuelas estatales vs % NBI', fontsize=12, fontweight='bold')

# Escuelas privadas vs NBI
axes[1].scatter(df_con_datos['pct_nbi'], df_con_datos['escuelas_privadas'],
                alpha=0.5, color='#e74c3c', s=30)
axes[1].set_xlabel('% NBI del barrio')
axes[1].set_ylabel('Cantidad de establecimientos privados')
axes[1].set_title('Escuelas privadas vs % NBI', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../tmp/exploracion_escuelas_nbi.png', dpi=130, bbox_inches='tight')
plt.show()

# Correlación
corr_estatal = df_con_datos[['pct_nbi','escuelas_estatales']].corr().iloc[0,1]
corr_privado = df_con_datos[['pct_nbi','escuelas_privadas']].corr().iloc[0,1]
print(f'Correlación pct_nbi vs escuelas estatales : {corr_estatal:+.3f}')
print(f'Correlación pct_nbi vs escuelas privadas  : {corr_privado:+.3f}')
print()
print('📌 Interpretación:')
print('  - Correlación negativa con privadas → más escuelas privadas en barrios con menos pobreza')
print('  - Correlación cerca de 0 con estatales → el Estado distribuye sin distinción de NBI')

## 6. Matriz de correlación entre variables

¿Qué variables están relacionadas entre sí?

In [ ]:
cols_num = ['poblacion', 'pct_nbi', 'escuelas_total', 'escuelas_estatales',
            'escuelas_privadas', 'centros_salud', 'paradas_colectivo', 
            'luminarias_reportes', 'comisarias']
cols_num = [c for c in cols_num if c in df.columns]

corr = df[cols_num].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im)
labels = [c.replace('_', '\n') for c in cols_num]
ax.set_xticks(range(len(cols_num)))
ax.set_yticks(range(len(cols_num)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
for i in range(len(cols_num)):
    for j in range(len(cols_num)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=7,
                color='white' if abs(corr.iloc[i,j]) > 0.5 else 'black')
ax.set_title('Matriz de Correlación — Variables de Desigualdad Urbana', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../tmp/exploracion_correlacion.png', dpi=130, bbox_inches='tight')
plt.show()

## 📝 Conclusiones del Análisis Exploratorio

1. **Distribución del NBI:** La pobreza se distribuye de forma asimétrica — la mayoría de barrios tiene NBI bajo, pero hay una cola de barrios muy vulnerables.
2. **Cobertura desigual:** La luminaria tiene la mayor cobertura (~65%). El transporte y la salud cubren ~30%. Las escuelas (con datos IDECOR) mejoran notablemente la cobertura.
3. **Escuelas estatales vs privadas:** Las privadas se concentran en barrios de menor NBI (correlación negativa). Las estatales no muestran esta diferencia.
4. **Para el siguiente notebook:** Agrupamos barrios por perfil socioeconómico usando clustering.